# Convert LoRA adapters to GGUF for Ollama

**Base model**: `Qwen/Qwen2.5-0.5B-Instruct`  
**Adapters**: `techcorp-medical-lora`  
**Output**: `techcorp-medical.gguf` — ready for Ollama

---

**Steps:**
1. Install dependencies
2. Upload your `techcorp-medical-lora.zip`
3. Merge LoRA with base model
4. Convert to GGUF
5. Download the `.gguf` file
6. Run `ollama create` + `ollama push` locally

## 1. Install dependencies

In [ ]:
%%capture
!pip install -q -U transformers peft accelerate
!pip install -q sentencepiece protobuf

# Clone llama.cpp for GGUF conversion
!git clone -q https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt

print('Done.')

## 2. Upload the LoRA zip

In [ ]:
from google.colab import files
import zipfile, os

print('Upload techcorp-medical-lora.zip ...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('lora_adapters')

# Find the adapter directory
for root, dirs, files_list in os.walk('lora_adapters'):
    if 'adapter_config.json' in files_list:
        ADAPTER_PATH = root
        break

print(f'Adapters found at: {ADAPTER_PATH}')
print(os.listdir(ADAPTER_PATH))

## 3. Merge LoRA adapters with base model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
MERGED_PATH = './merged_model'

print(f'Loading base model: {BASE_MODEL} ...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

print('Applying LoRA adapters ...')
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

print('Merging ...')
model = model.merge_and_unload()

print('Saving merged model ...')
model.save_pretrained(MERGED_PATH)
AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True).save_pretrained(MERGED_PATH)

print(f'Merged model saved to {MERGED_PATH}')
print(os.listdir(MERGED_PATH))

## 4. Convert to GGUF (Q4_K_M quantization)

In [ ]:
GGUF_OUTPUT = './techcorp-medical.gguf'

print('Converting to GGUF ...')
!python llama.cpp/convert_hf_to_gguf.py {MERGED_PATH} \
    --outfile {GGUF_OUTPUT} \
    --outtype q4_k_m

import os
size = os.path.getsize(GGUF_OUTPUT) / 1e6
print(f'GGUF file: {GGUF_OUTPUT} ({size:.1f} MB)')

## 5. Download the GGUF file

In [ ]:
from google.colab import files
print('Downloading techcorp-medical.gguf ...')
files.download('./techcorp-medical.gguf')

## 6. Upload to Ollama (run locally on your machine)

Once the `.gguf` is downloaded, run these commands in your **Windows terminal**:

```
ollama create liliaquispelopez/techcorp-medical -f Modelfile
ollama push liliaquispelopez/techcorp-medical
```

With this `Modelfile`:
```
FROM ./techcorp-medical.gguf

SYSTEM """
You are a medical assistant. Provide accurate, helpful information about symptoms,
medications, and general health. Always recommend consulting a qualified doctor
for diagnosis and treatment.
"""

PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER num_predict 512
PARAMETER repeat_penalty 1.1
```

> The model will be available at: https://ollama.com/liliaquispelopez/techcorp-medical